# 05 — RQ3 (Step 1): does network structure add signal over node features?

**Question (RQ3, Option B).** Same outcome and split as RQ2 (remixed within one year,
temporal train/test). Instead of asking "can a graph model predict remixing?", we ask the
sharper thesis question: **does a track's position in the remix network add predictive
signal beyond the 51 node/author/content features already used in `04`?**

**Staged plan.** Step 1 (this notebook) uses cheap, interpretable *structural* graph
features and a head-to-head model comparison to get a fast yes/no. Step 2 (a full GNN) is
only worth building if Step 1 shows a real lift.

**The leakage rule — stricter than RQ2.** A graph model can spread information across the
network, so a track must be described using **only edges that existed before it was posted**.
We enforce this with yearly *as-of snapshots*: for a track posted in year Y, its graph
features are read from a network built solely from remix edges whose child was posted before
1 January Y. A track's own remixes (its outcome) are therefore never visible. A brand-new
track has no realised edges except to its *sources*, so graph position is aggregated over a
track's **parent (source) tracks**.

In [1]:
import numpy as np, pandas as pd, networkx as nx
feat=pd.read_csv("data/processed/features.csv")
nc=pd.read_csv("data/processed/nodes_clean.csv")[["upload_id","date_unix"]]
loc=pd.read_csv("data/processed/edges.csv"); loc=loc[loc.edge_type=="local"].copy()
df=feat.merge(nc,on="upload_id"); df["t"]=df.date_unix.astype("int64")
df["pyear"]=pd.to_datetime(df.t,unit="s",utc=True).dt.year
idset=set(df.upload_id)
loc=loc[loc.parent_id.isin(idset)&loc.child_id.isin(idset)].sort_values("child_date_unix")
child2parents=loc.groupby("child_id").parent_id.apply(list).to_dict()
print("tracks",len(df),"| on-site local edges",len(loc))

tracks 51486 | on-site local edges 59186


## 1. As-of snapshots and structural metrics

For each year boundary we build the remix graph from edges realised before it, and compute
per node: PageRank (global centrality), in/out degree, k-core number, and connected-component
size. These are then aggregated over each track's parents.

In [2]:
years=sorted(df.pyear.unique()); metrics_by_year={}
for Y in years:
    b=int(pd.Timestamp(f"{Y}-01-01",tz="UTC").timestamp())
    sub=loc[loc.child_date_unix<b]
    if len(sub)==0: metrics_by_year[Y]=None; continue
    G=nx.from_pandas_edgelist(sub,"parent_id","child_id",create_using=nx.DiGraph())
    U=G.to_undirected(); comp={}
    for cc in nx.connected_components(U):
        for n in cc: comp[n]=len(cc)
    metrics_by_year[Y]=pd.DataFrame({"pr":nx.pagerank(G,alpha=0.85,max_iter=100),
        "ind":dict(G.in_degree()),"outd":dict(G.out_degree()),
        "core":nx.core_number(U),"comp":comp})

cols=["f_g_par_pr_max","f_g_par_pr_mean","f_g_par_outd_max","f_g_par_outd_mean",
      "f_g_par_ind_mean","f_g_par_core_max","f_g_par_comp_max","f_g_par_comp_mean","f_g_has_parent"]
def pf(uid,y):
    m=metrics_by_year.get(y); ps=child2parents.get(uid)
    if m is None or not ps: return (0,)*9
    s=m.reindex([p for p in ps if p in m.index]).dropna()
    if len(s)==0: return (0,)*9
    return (s.pr.max(),s.pr.mean(),s.outd.max(),s.outd.mean(),s.ind.mean(),
            int(s.core.max()),int(s.comp.max()),s.comp.mean(),1)
df=pd.concat([df,pd.DataFrame([pf(u,y) for u,y in zip(df.upload_id,df.pyear)],
                              columns=cols,index=df.index)],axis=1)
print("built graph features; tracks with a parent in the as-of graph:",round(df.f_g_has_parent.mean(),3))

built graph features; tracks with a parent in the as-of graph: 0.311


## 2. Leakage spot-check

A track's parent-PageRank must equal the value in its post-year snapshot, and that snapshot
must contain no edge dated on/after the boundary.

In [3]:
ex=df[(df.pyear==2015)&(df.f_g_has_parent==1)].iloc[0]
m=metrics_by_year[2015]; present=[p for p in child2parents[ex.upload_id] if p in m.index]
assert abs(ex.f_g_par_pr_max-m.reindex(present).pr.max())<1e-9
b=int(pd.Timestamp("2015-01-01",tz="UTC").timestamp())
assert (loc[loc.child_date_unix<b].child_date_unix<b).all()
print("spot-check passed: graph features use only pre-post-year edges")

spot-check passed: graph features use only pre-post-year edges


## 3. Head-to-head — node vs graph vs combined

Same tuned model, temporal test, primary (1-year) horizon. A paired bootstrap tests whether
node+graph reliably beats node-only.

In [4]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.inspection import permutation_importance
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
rng=np.random.default_rng(0)
graph_f=[c for c in df.columns if c.startswith("f_g_")]
node_f =[c for c in df.columns if c.startswith("f_") and c not in graph_f]
P=dict(max_depth=3,learning_rate=0.05,max_iter=300,l2_regularization=1.0,random_state=0)
tr=df[(df.split=="train")&df.valid_365]; te=df[(df.split=="test")&df.valid_365]
ytr=tr.y_365.astype(int).values; yt=te.y_365.astype(int).values
def ci(y,p,n=1000):
    ix=np.arange(len(y)); v=[roc_auc_score(y[s],p[s]) for s in (rng.choice(ix,len(ix),True) for _ in range(n)) if 0<y[s].sum()<len(s)]
    return np.percentile(v,[2.5,97.5])
def run(cols,lab):
    m=HistGradientBoostingClassifier(**P).fit(tr[cols].values,ytr); p=m.predict_proba(te[cols].values)[:,1]
    lo,hi=ci(yt,p); print(f"{lab:28s} feats={len(cols):2d} AUC={roc_auc_score(yt,p):.3f} [{lo:.3f},{hi:.3f}] AP={average_precision_score(yt,p):.3f}")
    return m,p,(roc_auc_score(yt,p),lo,hi)
print(f"test n={len(te)} base={yt.mean():.3f}")
mA,pA,rA=run(node_f,"A: node only (RQ2)")
mB,pB,rB=run(graph_f,"B: graph only")
mC,pC,rC=run(node_f+graph_f,"C: node + graph")
ix=np.arange(len(yt)); d=[roc_auc_score(yt[s],pC[s])-roc_auc_score(yt[s],pA[s]) for s in (rng.choice(ix,len(ix),True) for _ in range(2000)) if 0<yt[s].sum()<len(s)]
lo,hi=np.percentile(d,[2.5,97.5])
print(f"\npaired lift C-A: {np.mean(d):+.4f}  95%CI [{lo:+.4f},{hi:+.4f}]  -> {'adds signal' if lo>0 else 'no reliable gain'}")
pi=permutation_importance(mC,te[node_f+graph_f].values,yt,scoring="roc_auc",n_repeats=10,random_state=0)
imp=pd.Series(pi.importances_mean,index=node_f+graph_f).sort_values(ascending=False)
print("best graph-feature rank:",[i for i,c in enumerate(imp.index) if c in graph_f][0]+1,"of",len(imp))

fig,ax=plt.subplots(figsize=(6,4))
labs=["node\n(RQ2)","graph\nonly","node+graph"]; r=[rA,rB,rC]
ax.bar(labs,[x[0] for x in r],yerr=[[x[0]-x[1] for x in r],[x[2]-x[0] for x in r]],capsize=5,color=["#4c72b0","#c44e52","#55a868"])
ax.axhline(0.5,ls="--",c="grey",lw=1); ax.set_ylim(0.4,0.9); ax.set_ylabel("Test AUC (95% CI)")
ax.set_title("RQ3 Step 1: graph structure vs node features")
plt.tight_layout(); plt.savefig("fig_05_rq3.png",dpi=130); print("saved fig_05_rq3.png")

test n=3330 base=0.331
A: node only (RQ2)           feats=51 AUC=0.842 [0.829,0.855] AP=0.685
B: graph only                feats= 9 AUC=0.670 [0.654,0.685] AP=0.447
C: node + graph              feats=60 AUC=0.842 [0.828,0.856] AP=0.687

paired lift C-A: +0.0001  95%CI [-0.0012,+0.0015]  -> no reliable gain
best graph-feature rank: 9 of 60
saved fig_05_rq3.png


## 4. Conclusion

**Result.**
- Graph structure alone is predictive: AUC 0.670 [0.654, 0.685], well above chance.
- It adds **no reliable signal on top of the node features.** Node + graph gives AUC 0.842 [0.828, 0.856], the same as node-only at 0.842 [0.829, 0.855].
- The paired bootstrap lift is +0.0001 [−0.0012, +0.0015], an interval that includes zero.
- The best graph feature ranks only 9th of 60 in the combined model.

**Why this makes sense.** The predictive content of a track's network position is already captured by the simpler node features built in `03`. The sources' prior remix counts and the author's history act as local stand-ins for PageRank and coreness, so the graph adds only a redundant re-encoding.

**Caveats, and how they were addressed.**
1. **Coarse snapshots:** yearly snapshots are coarse, and only 31.1 % of tracks have a parent in the as-of graph.
2. **Hand-crafted metrics:** such metrics might miss higher-order structure.

`05b` tests both, with half-year snapshots and learned node2vec summaries, and the null result holds. On this basis a full GNN was not built.